In [1]:

# Question 1: Rwanda Maize Yield Dataset Preparation

## Stage 3: Explore and Audit the Data (EDA)

Group Number: Group 2

Members: Armand Ikirezi and Mariam Nyirarukundo

Scenario: Predicting Maize Yield for Farmer Cooperatives in Rwanda

Stakeholder: Maize Cooperative Support Unit (MCSU)

### Objective

To explore and audit the supplied maize yield dataset, identify data quality problems, examine variable distributions and relationships, and prepare a Data Quality Report.

Important: No data cleaning or modification will be performed during this stage.


SyntaxError: invalid syntax (2063343432.py, line 5)

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set the project directory
project_dir = Path(r"D:\MYProject\ML_Element1")

# Locate the original CSV
data_path = (
    project_dir
    / "q1_maize_yield"
    / "data"
    / "raw"
    / "maize_yield_rwanda_rw.csv"
)

# Load the original dataset
df = pd.read_csv(data_path)

# Display confirmation
print("Raw dataset loaded successfully!")
print("Dataset shape:", df.shape)


In [ ]:

# Display dataset dimensions
print("DATASET DIMENSIONS")
print("==================")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

# Display column information
print("\nDATASET INFORMATION")
print("===================")
df.info()

# Display numerical summary statistics
print("\nNUMERICAL SUMMARY STATISTICS")
print("============================")
display(df.describe())


In [ ]:

# Count missing values per column
missing_counts = df.isnull().sum()

# Calculate missing percentages
missing_percentages = (
    df.isnull().mean() * 100
)

# Create a missing-value summary
missing_report = pd.DataFrame({
    "Missing_Count": missing_counts,
    "Missing_Percentage": missing_percentages
})

# Display only columns with missing values
missing_report = missing_report[
    missing_report["Missing_Count"] > 0
].sort_values(
    by="Missing_Count",
    ascending=False
)

display(missing_report)

print("Total missing cells:", df.isnull().sum().sum())


In [ ]:
The dataset contains 39 missing values across three numerical columns: rainfall, soil pH and pest incidence index. 
These missing observations may affect statistical analysis and future model preparation. 
Their appropriate treatment will be determined during Stage 4.

In [ ]:

# Count complete duplicate rows
duplicate_rows = df.duplicated().sum()

# Count duplicated record IDs
duplicate_ids = df["record_id"].duplicated().sum()

print("DUPLICATE ANALYSIS")
print("==================")
print("Duplicate complete rows:", duplicate_rows)
print("Duplicate record IDs:", duplicate_ids)


In [ ]:
The duplicate analysis identified no complete duplicate rows and no repeated record identifiers. 
Therefore, no exact duplicate records were detected using these two checks.

In [ ]:

# Define checks for clearly invalid numerical values

invalid_checks = {
    "negative_rainfall": df["rainfall_mm"] < 0,
    "negative_altitude": df["altitude_m"] < 0,
    "invalid_soil_ph": (
        (df["soil_ph"] < 0) |
        (df["soil_ph"] > 14)
    ),
    "negative_temperature": df["avg_temp_c"] < 0,
    "negative_npk": df["npk_kg_ha"] < 0,
    "negative_urea": df["urea_kg_ha"] < 0,
    "negative_manure": df["manure_t_ha"] < 0,
    "nonpositive_farm_area": df["farm_area_ha"] <= 0,
    "nonpositive_plant_density": (
        df["plant_density_per_ha"] <= 0
    ),
    "nonpositive_yield": df["yield_t_ha"] <= 0
}

print("INVALID VALUE CHECKS")
print("=====================")

for check_name, condition in invalid_checks.items():
    print(f"{check_name}: {condition.sum()}")


In [ ]:

# Select numerical columns
numeric_columns = df.select_dtypes(
    include=np.number
).columns

# Calculate IQR outliers
outlier_results = []

for column in numeric_columns:

    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[
        (df[column] < lower_bound) |
        (df[column] > upper_bound)
    ][column].count()

    outlier_results.append({
        "Column": column,
        "Lower_Bound": lower_bound,
        "Upper_Bound": upper_bound,
        "Outlier_Count": outliers
    })

outlier_report = pd.DataFrame(outlier_results)

display(outlier_report)


In [ ]:

# Summary of the target variable
target = "yield_t_ha"

print("TARGET VARIABLE SUMMARY")
print("========================")
print(df[target].describe())

print("\nMissing target values:")
print(df[target].isnull().sum())


In [ ]:

plt.figure(figsize=(9, 5))

sns.histplot(
    data=df,
    x="yield_t_ha",
    bins=25,
    kde=True
)

plt.title("Distribution of Maize Yield")
plt.xlabel("Yield (tonnes per hectare)")
plt.ylabel("Number of farms")

plt.tight_layout()
plt.show()


In [ ]:

plt.figure(figsize=(9, 5))

sns.histplot(
    data=df,
    x="rainfall_mm",
    bins=25,
    kde=True
)

plt.title("Distribution of Rainfall")
plt.xlabel("Rainfall (mm)")
plt.ylabel("Number of records")

plt.tight_layout()
plt.show()


In [ ]:
The rainfall histogram shows how rainfall measurements are distributed across the records. 
The shape and spread of the distribution help identify unusually low or high rainfall observations that may require further investigation.

In [ ]:

province_yield = (
    df.groupby("province")["yield_t_ha"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

plt.figure(figsize=(10, 5))

sns.barplot(
    data=province_yield,
    x="province",
    y="yield_t_ha"
)

plt.title("Average Maize Yield by Province")
plt.xlabel("Province")
plt.ylabel("Average yield (tonnes/ha)")
plt.xticks(rotation=30)

plt.tight_layout()
plt.show()


In [ ]:
The bar chart compares average maize yield across provinces. 
It helps identify geographical differences in recorded yield, which may be associated with differences in environmental conditions and farming practices. 
These are descriptive differences, not proof of causation.

In [ ]:

plt.figure(figsize=(8, 5))

sns.boxplot(
    data=df,
    x="season",
    y="yield_t_ha"
)

plt.title("Maize Yield Distribution by Season")
plt.xlabel("Agricultural season")
plt.ylabel("Yield (tonnes/ha)")

plt.tight_layout()
plt.show()


In [ ]:
The box plot compares the distribution of maize yield across agricultural seasons. 
Differences in the median, spread and potential outliers may indicate seasonal variation in maize production.

In [ ]:

plt.figure(figsize=(9, 5))

sns.scatterplot(
    data=df,
    x="rainfall_mm",
    y="yield_t_ha",
    alpha=0.6
)

plt.title("Relationship Between Rainfall and Maize Yield")
plt.xlabel("Rainfall (mm)")
plt.ylabel("Yield (tonnes/ha)")

plt.tight_layout()
plt.show()


In [ ]:
The scatter plot explores the relationship between rainfall and maize yield. 
The pattern of points helps determine whether higher rainfall is associated with higher or lower recorded yields, or whether the relationship appears weak.

In [ ]:

categorical_columns = df.select_dtypes(
    include=["str", "object"]
).columns

for column in categorical_columns:

    original_values = df[column].dropna().astype(str)

    normalized_values = (
        original_values
        .str.strip()
        .str.lower()
    )

    print(f"\nColumn: {column}")
    print("Original unique values:", original_values.nunique())
    print("Normalized unique values:", normalized_values.nunique())

    if original_values.nunique() != normalized_values.nunique():
        print("Potential formatting inconsistencies detected.")
    else:
        print("No case/whitespace differences detected.")


In [ ]:

# Prepare the Data Quality Report

quality_issues = []

# 1. Missing values
for column, count in df.isnull().sum().items():
    if count > 0:
        quality_issues.append({
            "Problem": "Missing values",
            "Affected_Columns": column,
            "Records_Affected": int(count),
            "Proposed_Treatment": (
                "Investigate missingness and consider "
                "appropriate imputation in Stage 4."
            )
        })

# 2. Duplicate complete rows
quality_issues.append({
    "Problem": "Duplicate complete rows",
    "Affected_Columns": "All columns",
    "Records_Affected": int(df.duplicated().sum()),
    "Proposed_Treatment": (
        "Investigate duplicates; remove only if confirmed."
    )
})

# 3. Duplicate record IDs
quality_issues.append({
    "Problem": "Duplicate record identifiers",
    "Affected_Columns": "record_id",
    "Records_Affected": int(df["record_id"].duplicated().sum()),
    "Proposed_Treatment": (
        "Investigate repeated identifiers and determine "
        "whether they represent separate observations."
    )
})

# 4. Invalid values
for name, condition in invalid_checks.items():
    count = int(condition.sum())

    if count > 0:
        quality_issues.append({
            "Problem": f"Invalid values: {name}",
            "Affected_Columns": name,
            "Records_Affected": count,
            "Proposed_Treatment": (
                "Verify against source information and correct "
                "or handle in Stage 4 if confirmed invalid."
            )
        })

# 5. Potential IQR outliers
for _, row in outlier_report.iterrows():

    if row["Outlier_Count"] > 0:
        quality_issues.append({
            "Problem": "Potential IQR outliers",
            "Affected_Columns": row["Column"],
            "Records_Affected": int(row["Outlier_Count"]),
            "Proposed_Treatment": (
                "Investigate observations; retain genuine "
                "values and handle confirmed errors appropriately."
            )
        })

# 6. Potential categorical formatting issues
for column in categorical_columns:

    values = df[column].dropna().astype(str)

    normalized = values.str.strip().str.lower()

    if values.nunique() != normalized.nunique():

        quality_issues.append({
            "Problem": "Potential category formatting inconsistency",
            "Affected_Columns": column,
            "Records_Affected": "Requires detailed inspection",
            "Proposed_Treatment": (
                "Inspect unique values and standardize "
                "confirmed equivalent categories in Stage 4."
            )
        })

# Create the report
data_quality_report = pd.DataFrame(quality_issues)

display(data_quality_report)


In [ ]:

report_path = (
    project_dir
    / "q1_maize_yield"
    / "docs"
    / "data_quality_report.csv"
)

data_quality_report.to_csv(
    report_path,
    index=False
)

print("Data Quality Report saved successfully!")
print(report_path)
